In [ ]:
!pip install censusdatadownloader -q
!pip install geopandas -q
!pip install pygris -q          # fast Census tract shapefile downloader
!pip install sodapy -q          # Socrata API client for data.ny.gov
print('✅  Dependencies installed.')

ERROR: Could not find a version that satisfies the requirement censusdatadownloader (from versions: none)
ERROR: No matching distribution found for censusdatadownloader
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.2 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_i

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅  Drive mounted.')

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import warnings, os, math
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 60)

# ── PATHS — match exactly what the DiD notebook uses ─────────────────────────
PANEL_PATH = '/content/drive/MyDrive/MTA/clean_data/mta_crz_did_panel_raw_hourly_2024_2025.csv'
OUT_DIR    = '/content/drive/MyDrive/MTA/clean_data/'

# Sub-folders for raw downloads (create if missing)
ACS_DIR    = '/content/drive/MyDrive/MTA/ACS/'
BT_DIR     = '/content/drive/MyDrive/MTA/Bridges/'
for d in [ACS_DIR, BT_DIR]:
    os.makedirs(d, exist_ok=True)

# Output files
OUT_STATION_CSV = OUT_DIR + 'mta_crz_acs_station_joined.csv'
OUT_BT_CSV      = OUT_DIR + 'mta_crz_bridges_tunnels_daily.csv'

# ACS config
ACS_YEAR  = 2023          # most recent 5-year ACS available
ACS_STATE = '36'          # New York FIPS code
# B08201: household vehicle availability
# B08201_001E = total households
# B08201_002E = no vehicle available

# Census API key — get a free one at https://api.census.gov/data/key_signup.html
# Leave as None to use the unauthenticated endpoint (rate-limited but works)
CENSUS_API_KEY = None   # ← paste your key here if you have one: 'abc123...'

# Bridges & Tunnels Socrata dataset ID on data.ny.gov
BT_DATASET_ID = 'qzve-kjga'   # MTA Bridges and Tunnels Daily Traffic

def section(title):
    print(f"\n{'='*65}\n  {title}\n{'='*65}")

def check(condition, msg_fail, msg_pass=None):
    if not condition:
        raise AssertionError(f'\n❌  {msg_fail}')
    if msg_pass:
        print(f'  ✅  {msg_pass}')

print('✅  Config loaded.')
print(f'    Panel path : {PANEL_PATH}')
print(f'    Output dir : {OUT_DIR}')
print(f'    ACS year   : {ACS_YEAR}')

In [ ]:
section('1 · LOAD STATION METADATA')

# Check if df is already in memory from the DiD notebook session
try:
    _ = df
    print(f'  df already in memory — {len(df):,} rows. Extracting station metadata...')
    stations = (
        df.groupby('station_complex_id')
        .agg(
            station_complex = ('station_complex', 'first'),
            borough         = ('borough',         'first'),
            CRZ_Zone        = ('CRZ_Zone',        'first'),
            treated         = ('treated',          'first'),
            latitude        = ('latitude',         'first'),
            longitude       = ('longitude',        'first'),
        )
        .reset_index()
    )
    # Ensure station_complex_id is str (same fix as DiD notebook)
    stations['station_complex_id'] = stations['station_complex_id'].astype(str)

except NameError:
    # df not in memory — load just the columns we need to save time
    print('  df not in memory — loading station columns only (~1–2 min)...')
    STATION_COLS = [
        'station_complex_id', 'station_complex', 'borough',
        'CRZ_Zone', 'treated', 'latitude', 'longitude'
    ]
    raw = pd.read_csv(PANEL_PATH, usecols=STATION_COLS)
    raw['station_complex_id'] = raw['station_complex_id'].astype(str)
    raw = raw[raw['station_complex_id'] != '222']  # drop Roosevelt Island duplicate
    stations = raw.drop_duplicates(subset='station_complex_id').reset_index(drop=True)

# Sanity
check(len(stations) == 428,
      f'Expected 428 stations, got {len(stations)}.',
      f'Station count = {len(stations)} ✓')
check(stations['latitude'].isnull().sum() == 0,
      'Null latitudes in station metadata.',
      'No null lat/lon values ✓')

print(f'  Treated stations  : {(stations["treated"]==1).sum()}')
print(f'  Control stations  : {(stations["treated"]==0).sum()}')
print(f'  Lat range         : {stations["latitude"].min():.4f} → {stations["latitude"].max():.4f}')
print(f'  Lon range         : {stations["longitude"].min():.4f} → {stations["longitude"].max():.4f}')
print('\n  ✅  Section 1 complete.')

In [ ]:
section('2a · ACS — DOWNLOAD B08201 FROM CENSUS API')

# Variables:
# B08201_001E = total households
# B08201_002E = no vehicle available
# NAME        = tract name (for QA)

ACS_VARS = 'B08201_001E,B08201_002E,NAME'
ACS_GEO  = 'tract:*'
ACS_IN   = f'state:{ACS_STATE}&in=county:005,047,061,081,085'  # Bronx, Brooklyn, Manhattan, Queens, Staten Island FIPS

BASE_URL = f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5'

params = {
    'get' : ACS_VARS,
    'for' : ACS_GEO,
    'in'  : ACS_IN,
}
if CENSUS_API_KEY:
    params['key'] = CENSUS_API_KEY

print(f'  Requesting ACS {ACS_YEAR} 5-year estimates — Table B08201...')
print(f'  URL: {BASE_URL}')

resp = requests.get(BASE_URL, params=params, timeout=60)
check(resp.status_code == 200,
      f'Census API returned {resp.status_code}: {resp.text[:200]}',
      f'Census API responded OK (status {resp.status_code}) ✓')

data = resp.json()
acs_raw = pd.DataFrame(data[1:], columns=data[0])

# Type conversions
acs_raw['B08201_001E'] = pd.to_numeric(acs_raw['B08201_001E'], errors='coerce')
acs_raw['B08201_002E'] = pd.to_numeric(acs_raw['B08201_002E'], errors='coerce')

# Build GEOID for joining to shapefile
acs_raw['GEOID'] = acs_raw['state'] + acs_raw['county'] + acs_raw['tract']

# Car ownership rate: share of HH with ZERO vehicles (higher = fewer cars)
acs_raw['zero_vehicle_share'] = (
    acs_raw['B08201_002E'] / acs_raw['B08201_001E'].replace(0, np.nan)
)
# Car ownership rate: share of HH WITH at least one vehicle
acs_raw['car_ownership_rate'] = 1 - acs_raw['zero_vehicle_share']

# Save raw
acs_raw.to_csv(ACS_DIR + 'acs_b08201_raw.csv', index=False)

print(f'\n  Tracts downloaded  : {len(acs_raw):,}')
print(f'  Null total HH      : {acs_raw["B08201_001E"].isnull().sum()}')
print(f'  Null zero-veh      : {acs_raw["B08201_002E"].isnull().sum()}')
print(f'  Car ownership mean : {acs_raw["car_ownership_rate"].mean():.3f}')
print(f'  Car ownership range: {acs_raw["car_ownership_rate"].min():.3f} – {acs_raw["car_ownership_rate"].max():.3f}')
print(f'  Saved → {ACS_DIR}acs_b08201_raw.csv')
print('\n  ✅  Section 2a complete.')

In [ ]:
section('2b · ACS — LOAD CENSUS TRACT SHAPEFILE (pygris)')

from pygris import tracts

# NYC counties: 005=Bronx, 047=Brooklyn, 061=Manhattan, 081=Queens, 085=Staten Island
NYC_COUNTIES = ['005','047','061','081','085']

print('  Downloading NYC census tract shapefile from Census TIGER...')
tract_list = []
for county in NYC_COUNTIES:
    t = tracts(state=ACS_STATE, county=county, year=ACS_YEAR, cache=True)
    tract_list.append(t)
    print(f'    County {county}: {len(t)} tracts')

nyc_tracts = pd.concat(tract_list, ignore_index=True)
nyc_tracts = nyc_tracts.to_crs('EPSG:4326')  # WGS84 — same CRS as station lat/lon

print(f'\n  Total NYC tracts loaded : {len(nyc_tracts):,}')
print(f'  CRS                     : {nyc_tracts.crs}')

# Join ACS data onto shapefile by GEOID
nyc_tracts = nyc_tracts.merge(
    acs_raw[['GEOID','car_ownership_rate','zero_vehicle_share','B08201_001E']],
    on='GEOID', how='left'
)

matched = nyc_tracts['car_ownership_rate'].notna().sum()
print(f'  Tracts with ACS data     : {matched} / {len(nyc_tracts)}')
check(matched > len(nyc_tracts) * 0.9,
      f'Only {matched}/{len(nyc_tracts)} tracts matched ACS data — check GEOID format.',
      f'{matched}/{len(nyc_tracts)} tracts matched ACS data ✓')

print('\n  ✅  Section 2b complete.')

In [ ]:
section('2c · ACS — SPATIAL JOIN STATIONS → CENSUS TRACTS')

# Convert station lat/lon to GeoDataFrame points
from shapely.geometry import Point

stations_gdf = gpd.GeoDataFrame(
    stations.copy(),
    geometry=gpd.points_from_xy(stations['longitude'], stations['latitude']),
    crs='EPSG:4326'
)

print('  Running spatial join (station points → tract polygons)...')
stations_with_acs = gpd.sjoin(
    stations_gdf,
    nyc_tracts[['GEOID','car_ownership_rate','zero_vehicle_share','B08201_001E','geometry']],
    how='left',
    predicate='within'
)

# Drop geometry and sjoin index columns — we only need the ACS attributes
stations_with_acs = stations_with_acs.drop(columns=['geometry','index_right'], errors='ignore')
stations_with_acs = pd.DataFrame(stations_with_acs)  # back to plain DataFrame

# Dedup: some stations may match multiple tracts (edge cases) — keep first match
stations_with_acs = stations_with_acs.drop_duplicates(subset='station_complex_id').reset_index(drop=True)

# How many matched?
matched_acs = stations_with_acs['car_ownership_rate'].notna().sum()
unmatched   = stations_with_acs['car_ownership_rate'].isna().sum()

print(f'\n  Stations matched to a tract : {matched_acs} / {len(stations_with_acs)}')
print(f'  Unmatched (no tract found)  : {unmatched}')

if unmatched > 0:
    print('\n  Unmatched stations (will get borough-median imputation):')
    print(stations_with_acs[stations_with_acs['car_ownership_rate'].isna()]
          [['station_complex_id','station_complex','borough','latitude','longitude']]
          .to_string(index=False))

    # Impute unmatched with borough median car ownership rate
    borough_median = stations_with_acs.groupby('borough')['car_ownership_rate'].median()
    for idx, row in stations_with_acs[stations_with_acs['car_ownership_rate'].isna()].iterrows():
        b = row['borough']
        if b in borough_median.index:
            stations_with_acs.loc[idx, 'car_ownership_rate'] = borough_median[b]
            stations_with_acs.loc[idx, 'zero_vehicle_share'] = 1 - borough_median[b]
            stations_with_acs.loc[idx, 'acs_imputed'] = True

stations_with_acs['acs_imputed'] = stations_with_acs.get('acs_imputed', False).fillna(False)

print(f'\n  After imputation:')
print(f'  car_ownership_rate — mean  : {stations_with_acs["car_ownership_rate"].mean():.3f}')
print(f'  car_ownership_rate — range : {stations_with_acs["car_ownership_rate"].min():.3f} – {stations_with_acs["car_ownership_rate"].max():.3f}')
print(f'  Imputed stations           : {stations_with_acs["acs_imputed"].sum()}')

check(stations_with_acs['car_ownership_rate'].isnull().sum() == 0,
      'Still have null car_ownership_rate after imputation.',
      'No nulls in car_ownership_rate ✓')
check(len(stations_with_acs) == 428,
      f'Expected 428 stations after join, got {len(stations_with_acs)}.',
      f'Station count = {len(stations_with_acs)} ✓')

print('\n  ✅  Section 2c complete.')

In [ ]:
section('3a · BRIDGES & TUNNELS — DOWNLOAD FROM data.ny.gov')

# Socrata API — no key needed for read-only public datasets
# We pull the full dataset filtered to 2024–2025

BT_URL = f'https://data.ny.gov/resource/{BT_DATASET_ID}.json'

print(f'  Pulling from: {BT_URL}')
print('  Fetching in chunks of 50,000 rows...')

bt_frames = []
limit  = 50000
offset = 0

while True:
    params = {
        '$limit'  : limit,
        '$offset' : offset,
        '$where'  : "date >= '2024-01-01T00:00:00.000' AND date <= '2025-12-31T23:59:59.000'",
        '$order'  : 'date ASC',
    }
    resp = requests.get(BT_URL, params=params, timeout=120)
    check(resp.status_code == 200,
          f'Socrata API error {resp.status_code}: {resp.text[:300]}')

    batch = resp.json()
    if not batch:
        break

    bt_frames.append(pd.DataFrame(batch))
    print(f'    Fetched offset {offset:,} — {len(batch)} rows')
    offset += limit

    if len(batch) < limit:
        break

bt_raw = pd.concat(bt_frames, ignore_index=True)
print(f'\n  Total rows downloaded : {len(bt_raw):,}')
print(f'  Columns               : {list(bt_raw.columns)}')
print('\n  ✅  Section 3a complete.')

In [ ]:
section('3b · BRIDGES & TUNNELS — CLEAN + AGGREGATE TO DAILY TOTALS')

# Standardise column names to lowercase
bt_raw.columns = bt_raw.columns.str.lower().str.replace(' ','_')

print('  Columns after normalisation:')
print(f'    {list(bt_raw.columns)}')

# Identify the date column and vehicle count column
# Column names vary slightly by dataset version — find them dynamically
date_col    = next((c for c in bt_raw.columns if 'date' in c), None)
vehicle_col = next((c for c in bt_raw.columns if any(k in c for k in ['vehicle','count','volume','traffic'])), None)
plaza_col   = next((c for c in bt_raw.columns if any(k in c for k in ['plaza','crossing','facility','location'])), None)

print(f'\n  Date column    : {date_col}')
print(f'  Vehicle column : {vehicle_col}')
print(f'  Plaza column   : {plaza_col}')

check(date_col is not None, 'Could not find date column — check column names above.')
check(vehicle_col is not None, 'Could not find vehicle count column — check column names above.')

# Parse date and vehicle count
bt_raw['date_parsed']   = pd.to_datetime(bt_raw[date_col], errors='coerce')
bt_raw['vehicle_count'] = pd.to_numeric(bt_raw[vehicle_col], errors='coerce')

# Filter to 2024–2025
bt_raw = bt_raw[
    (bt_raw['date_parsed'] >= '2024-01-01') &
    (bt_raw['date_parsed'] <= '2025-12-31')
].copy()

# Aggregate to daily TOTAL across all plazas
bt_daily = (
    bt_raw.groupby(bt_raw['date_parsed'].dt.date)
    ['vehicle_count']
    .sum()
    .reset_index()
    .rename(columns={'date_parsed': 'Date', 'vehicle_count': 'bt_total_vehicles'})
)
bt_daily['Date'] = pd.to_datetime(bt_daily['Date'])

# Add post flag and rolling 7-day average
bt_daily['post']             = (bt_daily['Date'] >= '2025-01-05').astype(int)
bt_daily['bt_vehicles_7d']   = bt_daily['bt_total_vehicles'].rolling(7, center=True).mean()

# Save
bt_daily.to_csv(OUT_BT_CSV, index=False)

print(f'\n  Daily rows (2024–2025) : {len(bt_daily):,}')
print(f'  Date range             : {bt_daily["Date"].min().date()} → {bt_daily["Date"].max().date()}')
print(f'  Mean daily vehicles    : {bt_daily["bt_total_vehicles"].mean():,.0f}')

pre_mean  = bt_daily[bt_daily['post']==0]['bt_total_vehicles'].mean()
post_mean = bt_daily[bt_daily['post']==1]['bt_total_vehicles'].mean()
pct_drop  = (post_mean - pre_mean) / pre_mean * 100

print(f'\n  Pre-CRZ mean  (2024)  : {pre_mean:,.0f} vehicles/day')
print(f'  Post-CRZ mean (2025)  : {post_mean:,.0f} vehicles/day')
print(f'  % change              : {pct_drop:+.2f}%')

if pct_drop < 0:
    print('  ✅  Vehicle crossings dropped post-CRZ — consistent with modal shift hypothesis.')
else:
    print('  ⚠️   Vehicle crossings did NOT drop post-CRZ — inspect the data or date range.')

print(f'\n  Saved → {OUT_BT_CSV}')
print('\n  ✅  Section 3b complete.')

In [ ]:
section('4 · TOLL CROSSING COORDINATES → DISTANCE TO NEAREST PLAZA')

# ── 9 NYC bridge/tunnel toll crossing lat/lon ─────────────────────────────────
# Source: MTA map + Google Maps verification
TOLL_PLAZAS = pd.DataFrame([
    {'plaza': 'Queens-Midtown Tunnel',        'lat': 40.7446, 'lon': -73.9718},
    {'plaza': 'Lincoln Tunnel',               'lat': 40.7614, 'lon': -74.0024},
    {'plaza': 'Hugh L. Carey Tunnel (BBT)',   'lat': 40.6867, 'lon': -74.0148},
    {'plaza': 'Holland Tunnel',               'lat': 40.7272, 'lon': -74.0282},
    {'plaza': 'Verrazzano-Narrows Bridge',    'lat': 40.6066, 'lon': -74.0442},
    {'plaza': 'Goethals Bridge',              'lat': 40.6423, 'lon': -74.2006},
    {'plaza': 'Bayonne Bridge',               'lat': 40.6484, 'lon': -74.1445},
    {'plaza': 'George Washington Bridge',     'lat': 40.8517, 'lon': -73.9527},
    {'plaza': 'Robert F. Kennedy Bridge',     'lat': 40.7775, 'lon': -73.9200},
])

print('  Toll plaza coordinates:')
print(TOLL_PLAZAS.to_string(index=False))

# ── Haversine distance function ───────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

# ── Compute minimum distance from each station to any toll plaza ───────────────
print('\n  Computing haversine distances...')
dist_records = []
for _, stn in stations_with_acs.iterrows():
    min_dist = float('inf')
    nearest_plaza = ''
    for _, plaza in TOLL_PLAZAS.iterrows():
        d = haversine_km(stn['latitude'], stn['longitude'],
                         plaza['lat'], plaza['lon'])
        if d < min_dist:
            min_dist = d
            nearest_plaza = plaza['plaza']
    dist_records.append({
        'station_complex_id': stn['station_complex_id'],
        'dist_to_toll_km':    round(min_dist, 4),
        'nearest_plaza':      nearest_plaza,
    })

dist_df = pd.DataFrame(dist_records)

# Gateway station flag: within 0.8 km (~0.5 miles) of a toll plaza
GATEWAY_THRESHOLD_KM = 0.8
dist_df['is_gateway'] = (dist_df['dist_to_toll_km'] <= GATEWAY_THRESHOLD_KM).astype(int)

print(f'\n  Distance stats:')
print(f'  Min dist to toll  : {dist_df["dist_to_toll_km"].min():.3f} km')
print(f'  Max dist to toll  : {dist_df["dist_to_toll_km"].max():.3f} km')
print(f'  Mean dist to toll : {dist_df["dist_to_toll_km"].mean():.3f} km')
print(f'  Gateway stations (≤{GATEWAY_THRESHOLD_KM} km): {dist_df["is_gateway"].sum()}')

print('\n  Gateway stations:')
gw = dist_df[dist_df['is_gateway']==1].merge(
    stations_with_acs[['station_complex_id','station_complex','borough']], on='station_complex_id'
)
print(gw[['station_complex','borough','dist_to_toll_km','nearest_plaza']].to_string(index=False))

print('\n  ✅  Section 4 complete.')

In [ ]:
section('5 · MERGE ALL THREE → FINAL STATION FILE')

# Start with ACS-enriched stations (already has station metadata)
final = stations_with_acs.copy()

# Merge distance/gateway columns
final = final.merge(
    dist_df[['station_complex_id','dist_to_toll_km','nearest_plaza','is_gateway']],
    on='station_complex_id',
    how='left'
)

# Column audit
expected_cols = [
    'station_complex_id','station_complex','borough','CRZ_Zone',
    'treated','latitude','longitude',
    'GEOID','car_ownership_rate','zero_vehicle_share','B08201_001E','acs_imputed',
    'dist_to_toll_km','nearest_plaza','is_gateway',
]

missing = [c for c in expected_cols if c not in final.columns]
if missing:
    print(f'  ⚠️   Missing expected columns: {missing}')
else:
    print('  ✅  All expected columns present ✓')

# Final sanity checks
check(len(final) == 428, f'Expected 428 rows, got {len(final)}.', f'Row count = {len(final)} ✓')
check(final['car_ownership_rate'].isnull().sum() == 0, 'Nulls in car_ownership_rate.', 'No nulls in car_ownership_rate ✓')
check(final['dist_to_toll_km'].isnull().sum() == 0, 'Nulls in dist_to_toll_km.', 'No nulls in dist_to_toll_km ✓')
check(final['station_complex_id'].duplicated().sum() == 0, 'Duplicate station IDs.', 'No duplicate station IDs ✓')

# Save
final.to_csv(OUT_STATION_CSV, index=False)

print(f'\n  Final file shape : {final.shape}')
print(f'  Columns          : {list(final.columns)}')
print(f'\n  Saved → {OUT_STATION_CSV}')
print('\n  ✅  Section 5 complete.')

In [ ]:
section('6 · SANITY CHECKS + OUTPUT SUMMARY')

# ── ACS checks ────────────────────────────────────────────────────────────────
print('  ACS car ownership breakdown by borough:')
acs_borough = final.groupby('borough')['car_ownership_rate'].agg(['mean','min','max','count'])
print(acs_borough.round(3).to_string())

# Treated vs control car ownership — should differ if hypothesis holds
car_treated = final[final['treated']==1]['car_ownership_rate'].mean()
car_control = final[final['treated']==0]['car_ownership_rate'].mean()
print(f'\n  Mean car ownership — treated CBD  : {car_treated:.3f}')
print(f'  Mean car ownership — control outer: {car_control:.3f}')
print('  (Control outer-borough stations expected to have higher car ownership)')

# ── Gateway checks ────────────────────────────────────────────────────────────
print(f'\n  Gateway stations by borough:')
print(final.groupby(['borough','is_gateway']).size().unstack(fill_value=0).to_string())

# ── Bridges & Tunnels validation ──────────────────────────────────────────────
print(f'\n  Bridges & Tunnels daily traffic (2024 vs 2025):')
print(f'    Pre-CRZ mean  : {pre_mean:,.0f} vehicles/day')
print(f'    Post-CRZ mean : {post_mean:,.0f} vehicles/day')
print(f'    % change      : {pct_drop:+.2f}%')

# ── File confirmation ─────────────────────────────────────────────────────────
print('\n  Output files:')
for f in [OUT_STATION_CSV, OUT_BT_CSV]:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) / 1024 if exists else 0
    print(f'  {"✅" if exists else "❌ NOT FOUND"}  {f.split("/")[-1]}  ({size:.1f} KB)')

print('''
  ─────────────────────────────────────────────────────────────────
  OUTPUTS PRODUCED
  ─────────────────────────────────────────────────────────────────
  mta_crz_acs_station_joined.csv
    → 428 rows, one per station
    → columns: car_ownership_rate, zero_vehicle_share, GEOID,
               acs_imputed, dist_to_toll_km, nearest_plaza,
               is_gateway (+ all original station metadata)

  mta_crz_bridges_tunnels_daily.csv
    → one row per date, 2024–2025
    → columns: Date, bt_total_vehicles, post, bt_vehicles_7d

  ─────────────────────────────────────────────────────────────────
  HOW TO USE THESE IN THE DiD NOTEBOOK
  ─────────────────────────────────────────────────────────────────
  1. Join mta_crz_acs_station_joined.csv onto hourly panel:

     enrichment = pd.read_csv(OUT_STATION_CSV)
     hourly = hourly.merge(
         enrichment[['station_complex_id','car_ownership_rate',
                     'dist_to_toll_km','is_gateway']],
         on='station_complex_id', how='left'
     )

  2. Add to FORMULA_B:

     FORMULA_ENRICHED = (
         'log_ridership ~ did_term'
         ' + did_term:car_ownership_rate'   # moderation effect
         ' + did_term:dist_to_toll_km'      # proximity effect
         ' + daily_crz_entries'
         ' + fare_evasion_pct'
         ' + is_weekend'
         f' + {quarter_formula}'
         ' + EntityEffects'
     )

  3. Join mta_crz_bridges_tunnels_daily.csv onto panel by Date
     and add bt_total_vehicles as an additional control.

  ─────────────────────────────────────────────────────────────────
  NEXT STEP → Monte Carlo Simulation
  ─────────────────────────────────────────────────────────────────
''')